# ACC-HPC Hit RSA Linear Model

Model subject-level ACC->HPC hit RSA (`ACC_HPC_HitRSA_cluster_mean`) as a function of:

- ACC reinstatement mean accuracy (`ACC_MeanAccuracy`)
- ACC hit/miss HFA mean accuracy (`ACC_hfa_MeanAccuracy`)
- HPC hit/miss theta mean accuracy (`HPC_theta_MeanAccuracy`)
- HPC->ACC theta GC net z-score (`HPC_ACC_theta_gc_net_z_cue_full_pos`) computed in cue window 0-1 s for positive RPE

All predictors are z-scored before OLS.

In [ ]:
import os
import pickle
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from mne.stats import permutation_cluster_1samp_test
from scipy.sparse import csr_matrix, eye, kron
from scipy.stats import spearmanr

sns.set_style('white')


In [ ]:
# ---- Config ----
hit_memory_label = 'remembered'
rsa_metric_col = 'ACC_HPC_HitRSA_cluster_mean'

cluster_alpha = 0.05
cluster_n_permutations = 1000
cluster_seed = 42
cluster_tail = 1

rsa_group_results_pkl = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/RSA/crossregion/'
    'group_results_ACC_to_HPC.pkl'
)
acc_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)
acc_hfa_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'ACC_memory_outcome_hit_miss_hfa_subject_reinstatement.csv'
)
hpc_theta_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'HPC_memory_outcome_hit_miss_theta_subject_reinstatement.csv'
)
gc_merged_csv = (
    '/Volumes/T7 Shield/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_rpe/'
    'notebook_runs/cue_on_Day1_t0p0-1p0_sf50_4-8Hz_lag10_roie86d0eb437/merged_gc_rpe.csv'
)
pac_combined_csv = (
    '/Volumes/T7 Shield/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac/'
    'all_subjects_mv_gcpac_combined.csv'
)

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_hpc_rsa_linear_model'
)
figure_output_root = Path('/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots')
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

paths = {
    'merged_csv': analysis_output_root / 'acc_hpc_rsa_linear_model_merged_subject_table.csv',
    'coef_csv': analysis_output_root / 'acc_hpc_rsa_linear_model_coefficients.csv',
    'summary_txt': analysis_output_root / 'acc_hpc_rsa_linear_model_summary.txt',
    'corr_heatmap_pdf': figure_output_root / 'acc_hpc_rsa_linear_model_predictor_corr_heatmap.pdf',
    'diagnostic_pdf': figure_output_root / 'acc_hpc_rsa_linear_model_diagnostics.pdf',
    'pac_subject_csv': analysis_output_root / 'all_region_pac_subject_table.csv',
    'pac_coef_csv': analysis_output_root / 'all_region_pac_ols_coefficients.csv',
    'pac_summary_txt': analysis_output_root / 'all_region_pac_ols_summaries.txt',
    'pac_corr_pdf': figure_output_root / 'all_region_pac_outcome_predictor_correlation.pdf',
    'pac_diagnostic_pdf': figure_output_root / 'all_region_pac_ols_diagnostics.pdf',
    'pac_rsa_scatter_pdf': figure_output_root / 'hpc_acc_pac_overall_vs_acc_hpc_hitrsa_cluster_mean_scatter.pdf',
    'pac_rsa_scatter_hpc_hpc_pdf': figure_output_root / 'hpc_hpc_pac_overall_vs_acc_hpc_hitrsa_cluster_mean_scatter.pdf',
    'pac_rsa_scatter_acc_acc_pdf': figure_output_root / 'acc_acc_pac_overall_vs_acc_hpc_hitrsa_cluster_mean_scatter.pdf',
    'pac_overall_all_pairs_bar_strip_pdf': figure_output_root / 'all_pairs_pac_overall_cue_on_bar_strip.pdf',
}

print('RSA PKL:', rsa_group_results_pkl)
print('ACC reinstatement CSV:', acc_reinstatement_csv)
print('ACC hit_miss HFA CSV:', acc_hfa_reinstatement_csv)
print('HPC hit_miss theta CSV:', hpc_theta_reinstatement_csv)
print('GC merged CSV:', gc_merged_csv)
print('PAC combined CSV:', pac_combined_csv)
print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}')


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def zscore_series(s: pd.Series) -> pd.Series:
    arr = s.to_numpy(dtype=float)
    m = np.nanmean(arr)
    sd = np.nanstd(arr, ddof=0)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError(f'Cannot z-score series with zero/invalid std: {s.name}')
    return pd.Series((arr - m) / sd, index=s.index, name=f'{s.name}_z')


In [ ]:
# ---- Reconstruct significant ACC->HPC RSA cluster and build subject target ----
if not os.path.isfile(rsa_group_results_pkl):
    raise FileNotFoundError(f'RSA group results PKL not found: {rsa_group_results_pkl}')

with open(rsa_group_results_pkl, 'rb') as f:
    rsa_group_results = pickle.load(f)

required_rsa_keys = [
    'observed_diffs_stack',
    'feedback_times',
    'cue_times',
    'subject_results',
    'successful_subjects',
    'n_subjects',
]
missing_rsa_keys = [k for k in required_rsa_keys if k not in rsa_group_results]
if missing_rsa_keys:
    raise ValueError(f'RSA group results missing required keys: {missing_rsa_keys}')

X = np.asarray(rsa_group_results['observed_diffs_stack'], dtype=float)
feedback_times = np.asarray(rsa_group_results['feedback_times'], dtype=float)
cue_times = np.asarray(rsa_group_results['cue_times'], dtype=float)

if X.ndim != 3:
    raise ValueError(f'Expected observed_diffs_stack to be 3D; got shape {X.shape}.')
if X.shape[1] != len(feedback_times) or X.shape[2] != len(cue_times):
    raise ValueError(
        'Mismatch between observed_diffs_stack and feedback/cue time dimensions: '
        f'X={X.shape}, feedback={len(feedback_times)}, cue={len(cue_times)}'
    )

n_fb = X.shape[1]
n_cue = X.shape[2]

fb_adjacency = mne.stats.combine_adjacency(n_fb)
cue_adjacency = mne.stats.combine_adjacency(n_cue)
adjacency_2d = csr_matrix(kron(eye(n_fb), cue_adjacency) + kron(fb_adjacency, eye(n_cue)))
X_flat = X.reshape(X.shape[0], -1)

t_obs, clusters, cluster_pv, H0 = permutation_cluster_1samp_test(
    X_flat,
    n_permutations=cluster_n_permutations,
    threshold=None,
    adjacency=adjacency_2d,
    tail=cluster_tail,
    n_jobs=8,
    seed=cluster_seed,
    verbose=False,
)

sig_cluster_indices = [i for i, pval in enumerate(cluster_pv) if pval < cluster_alpha]
if len(sig_cluster_indices) != 1:
    raise ValueError(
        f'Expected exactly 1 significant ACC->HPC cluster at alpha={cluster_alpha}; '
        f'found {len(sig_cluster_indices)}.'
    )

sig_cluster_index = sig_cluster_indices[0]
sig_cluster = clusters[sig_cluster_index]
sig_mask_flat = np.zeros(n_fb * n_cue, dtype=bool)
sig_mask_flat[sig_cluster[0]] = True
sig_cluster_mask = sig_mask_flat.reshape(n_fb, n_cue)
sig_cluster_size = int(sig_cluster_mask.sum())

fb_indices = np.where(sig_cluster_mask.any(axis=1))[0]
cue_indices = np.where(sig_cluster_mask.any(axis=0))[0]
if len(fb_indices) == 0 or len(cue_indices) == 0:
    raise ValueError('Significant cluster mask is empty after reshaping.')

cluster_metadata = {
    'cluster_index_zero_based': int(sig_cluster_index),
    'cluster_p_value': float(cluster_pv[sig_cluster_index]),
    'cluster_size': sig_cluster_size,
    'n_cluster_subjects': int(X.shape[0]),
    'n_feedback_timepoints': int(n_fb),
    'n_cue_timepoints': int(n_cue),
    'feedback_tmin': float(feedback_times[fb_indices[0]]),
    'feedback_tmax': float(feedback_times[fb_indices[-1]]),
    'cue_tmin': float(cue_times[cue_indices[0]]),
    'cue_tmax': float(cue_times[cue_indices[-1]]),
    'n_feedback_rows_in_mask': int(len(fb_indices)),
    'n_cue_cols_in_mask': int(len(cue_indices)),
    'mask_bbox_size': int(len(fb_indices) * len(cue_indices)),
    'mask_fraction_in_bbox': float(sig_cluster_size / (len(fb_indices) * len(cue_indices))),
    'cluster_alpha': float(cluster_alpha),
    'cluster_n_permutations': int(cluster_n_permutations),
    'cluster_tail': int(cluster_tail),
    'cluster_seed': int(cluster_seed),
}
cluster_metadata_df = pd.DataFrame([cluster_metadata])

print('ACC->HPC cluster test complete.')
print('Significant clusters found:', len(sig_cluster_indices))
display(cluster_metadata_df)

rsa_rows = []
for subject_result in rsa_group_results['subject_results']:
    observed = subject_result['observed']
    subject = observed['subject']

    category_heatmaps = observed.get('category_heatmaps', {})
    if hit_memory_label not in category_heatmaps:
        raise ValueError(f'Subject {subject} missing category_heatmaps[{hit_memory_label!r}].')

    hit_heatmap = np.asarray(category_heatmaps[hit_memory_label], dtype=float)
    if hit_heatmap.shape != sig_cluster_mask.shape:
        raise ValueError(
            f'Subject {subject} heatmap shape {hit_heatmap.shape} does not match '
            f'cluster mask shape {sig_cluster_mask.shape}.'
        )

    diff_heatmap = np.asarray(observed['diff_heatmap'], dtype=float)
    n_trials_per_category = observed.get('n_trials_per_category', {})

    rsa_rows.append({
        'Subject': subject,
        rsa_metric_col: float(np.nanmean(hit_heatmap[sig_cluster_mask])),
        'ACC_HPC_DiffRSA_cluster_mean': float(np.nanmean(diff_heatmap[sig_cluster_mask])),
        'ACC_HPC_RememberedTrialCount': int(n_trials_per_category.get('remembered', 0)),
        'ACC_HPC_ForgottenTrialCount': int(n_trials_per_category.get('forgotten', 0)),
        'ACC_HPC_TotalTrialCount': int(observed.get('n_trials', 0)),
    })

rsa_subject_tbl = (
    pd.DataFrame(rsa_rows)
    .sort_values('Subject')
    .reset_index(drop=True)
)
assert_required_columns(
    rsa_subject_tbl,
    ['Subject', rsa_metric_col, 'ACC_HPC_RememberedTrialCount', 'ACC_HPC_ForgottenTrialCount'],
    'ACC-HPC hit RSA subject table',
)
print('Subjects in ACC->HPC hit RSA table:', rsa_subject_tbl['Subject'].nunique())
display(rsa_subject_tbl.head())


In [ ]:
# ---- Load model predictors ----
if not os.path.isfile(acc_reinstatement_csv):
    raise FileNotFoundError(f'ACC reinstatement CSV not found: {acc_reinstatement_csv}')
if not os.path.isfile(acc_hfa_reinstatement_csv):
    raise FileNotFoundError(f'ACC HFA hit_miss CSV not found: {acc_hfa_reinstatement_csv}')
if not os.path.isfile(hpc_theta_reinstatement_csv):
    raise FileNotFoundError(f'HPC theta hit_miss CSV not found: {hpc_theta_reinstatement_csv}')
if not os.path.isfile(gc_merged_csv):
    raise FileNotFoundError(f'GC merged CSV not found: {gc_merged_csv}')

acc_tbl = pd.read_csv(acc_reinstatement_csv)
acc_hfa_tbl = pd.read_csv(acc_hfa_reinstatement_csv)
hpc_theta_tbl = pd.read_csv(hpc_theta_reinstatement_csv)
gc_tbl = pd.read_csv(gc_merged_csv)

assert_required_columns(acc_tbl, ['Subject', 'MeanAccuracy', 'NumElectrodes'], 'ACC reinstatement table')
assert_required_columns(acc_hfa_tbl, ['Subject', 'MeanAccuracy', 'NumElectrodes'], 'ACC hit_miss HFA table')
assert_required_columns(hpc_theta_tbl, ['Subject', 'MeanAccuracy', 'NumElectrodes'], 'HPC hit_miss theta table')
assert_required_columns(gc_tbl, ['participant', 'roi_source', 'roi_target', 'rpe_cat', 'time', 'gc_net_z'], 'GC merged table')

acc_tbl = acc_tbl.rename(columns={'MeanAccuracy': 'ACC_MeanAccuracy', 'NumElectrodes': 'ACC_NumElectrodes'})[
    ['Subject', 'ACC_MeanAccuracy', 'ACC_NumElectrodes']
]
acc_hfa_tbl = acc_hfa_tbl.rename(columns={'MeanAccuracy': 'ACC_hfa_MeanAccuracy', 'NumElectrodes': 'ACC_hfa_NumElectrodes'})[
    ['Subject', 'ACC_hfa_MeanAccuracy', 'ACC_hfa_NumElectrodes']
]
hpc_theta_tbl = hpc_theta_tbl.rename(columns={'MeanAccuracy': 'HPC_theta_MeanAccuracy', 'NumElectrodes': 'HPC_theta_NumElectrodes'})[
    ['Subject', 'HPC_theta_MeanAccuracy', 'HPC_theta_NumElectrodes']
]

gc_mask = (
    (gc_tbl['roi_source'] == 'HPC')
    & (gc_tbl['roi_target'] == 'ACC')
    & (gc_tbl['rpe_cat'] == 'pos')
    & (gc_tbl['time'] >= 0.0)
    & (gc_tbl['time'] <= 1.0)
)
gc_subject_tbl = (
    gc_tbl.loc[gc_mask]
    .groupby('participant', as_index=False)['gc_net_z']
    .mean()
    .rename(columns={'participant': 'Subject', 'gc_net_z': 'HPC_ACC_theta_gc_net_z_cue_full_pos'})
    .sort_values('Subject')
    .reset_index(drop=True)
)

print('Subjects in ACC predictor table:', acc_tbl['Subject'].nunique())
print('Subjects in ACC HFA predictor table:', acc_hfa_tbl['Subject'].nunique())
print('Subjects in HPC theta predictor table:', hpc_theta_tbl['Subject'].nunique())
print('Subjects in GC predictor table:', gc_subject_tbl['Subject'].nunique())


In [ ]:
# ---- Merge modeling table and z-score predictors ----
model_tbl = deterministic_inner_join(rsa_subject_tbl, acc_tbl, keys=['Subject'])
model_tbl = deterministic_inner_join(model_tbl, acc_hfa_tbl, keys=['Subject'])
model_tbl = deterministic_inner_join(model_tbl, hpc_theta_tbl, keys=['Subject'])
model_tbl = deterministic_inner_join(model_tbl, gc_subject_tbl, keys=['Subject'])

if len(model_tbl) == 0:
    raise ValueError('No overlapping subjects across RSA target and predictors.')

predictor_cols = [
    'ACC_MeanAccuracy',
    'ACC_hfa_MeanAccuracy',
    'HPC_theta_MeanAccuracy',
    'HPC_ACC_theta_gc_net_z_cue_full_pos',
]
missing_audit = model_tbl[['Subject', rsa_metric_col] + predictor_cols].isna().sum()
if missing_audit.drop('Subject').sum() > 0:
    raise ValueError(f'Missing values present in model table:\n{missing_audit}')

for col in predictor_cols:
    model_tbl[f'{col}_z'] = zscore_series(model_tbl[col])

z_cols = [f'{c}_z' for c in predictor_cols]
if not np.isfinite(model_tbl[z_cols].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite values found in z-scored predictors.')

print('Subjects in final model table:', model_tbl['Subject'].nunique())
display(model_tbl[['Subject', rsa_metric_col] + predictor_cols + z_cols].head())


In [ ]:
# ---- Spearman correlations: RSA target vs individual predictors ----
x_cols = [
    'ACC_MeanAccuracy_z',
    'ACC_hfa_MeanAccuracy_z',
    'HPC_theta_MeanAccuracy_z',
    'HPC_ACC_theta_gc_net_z_cue_full_pos_z',
]
y_col = rsa_metric_col

coef_rows = []
for x_col in x_cols:
    corr_tbl = model_tbl[
        np.isfinite(model_tbl[x_col])
        & np.isfinite(model_tbl[y_col])
    ][['Subject', x_col, y_col]].copy()

    n_subjects = len(corr_tbl)
    if n_subjects < 3:
        coef_rows.append({
            'term': x_col,
            'rho': np.nan,
            'p_value': np.nan,
            'n_subjects': int(n_subjects),
            'status': 'too_few_points',
        })
        continue

    rho, p_value = spearmanr(
        corr_tbl[x_col].to_numpy(dtype=float),
        corr_tbl[y_col].to_numpy(dtype=float),
    )
    coef_rows.append({
        'term': x_col,
        'rho': float(rho),
        'p_value': float(p_value),
        'n_subjects': int(n_subjects),
        'status': 'ok',
    })

coef_df = pd.DataFrame(coef_rows)

corr_df = model_tbl[x_cols].corr(method='spearman')
fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)
sns.heatmap(corr_df, annot=True, cmap='vlag', center=0.0, square=True, ax=ax)
ax.set_title('Predictor Correlation Matrix (Spearman)')
fig.savefig(paths['corr_heatmap_pdf'], dpi=300)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
axes = np.asarray(axes).reshape(-1)
for i, x_col in enumerate(x_cols):
    scatter_tbl = model_tbl[
        np.isfinite(model_tbl[x_col])
        & np.isfinite(model_tbl[y_col])
    ][[x_col, y_col]].copy()

    sns.scatterplot(
        data=scatter_tbl,
        x=x_col,
        y=y_col,
        ax=axes[i],
        s=52,
        color='#4c78a8',
        edgecolor='black',
        linewidth=0.7,
    )
    axes[i].set_title(f'{y_col} vs {x_col}')
    axes[i].grid(False)
    sns.despine(ax=axes[i], top=True, right=True)

fig.savefig(paths['diagnostic_pdf'], dpi=300)
plt.close(fig)

model_tbl.to_csv(paths['merged_csv'], index=False)
coef_df.to_csv(paths['coef_csv'], index=False)
with open(paths['summary_txt'], 'w', encoding='utf-8') as f:
    f.write('ACC-HPC HIT RSA SPEARMAN CORRELATIONS\n')
    f.write('====================================\n\n')
    f.write(f'N subjects: {len(model_tbl)}\n')
    f.write(f'Target: {y_col}\n')
    f.write(f'Predictors: {x_cols}\n\n')
    f.write(coef_df.to_string(index=False))
    f.write('\n')

print('Saved merged model table:', paths['merged_csv'])
print('Saved Spearman results:', paths['coef_csv'])
print('Saved Spearman summary:', paths['summary_txt'])
print('Saved predictor correlation heatmap:', paths['corr_heatmap_pdf'])
print('Saved Spearman scatter grid:', paths['diagnostic_pdf'])
display(coef_df)


In [ ]:
# ---- Build subject-level PAC outcomes for every PAC ROI pair (cue_on; averaged across hemispheres) ----
if not os.path.isfile(pac_combined_csv):
    raise FileNotFoundError(f'PAC combined CSV not found: {pac_combined_csv}')

pac_tbl = pd.read_csv(pac_combined_csv)
assert_required_columns(
    pac_tbl,
    ['participant', 'phase_roi', 'amp_roi', 'event', 'rpe_cat', 'trial_pac_z', 'hemi'],
    'PAC combined table',
)

pac_tbl = pac_tbl.dropna(subset=['rpe_cat']).copy()
pac_tbl = pac_tbl[pac_tbl['event'] == 'cue_on'].copy()
if len(pac_tbl) == 0:
    raise ValueError('No cue_on PAC rows found after dropping missing rpe_cat.')

pair_keys = (
    pac_tbl[['phase_roi', 'amp_roi']]
    .drop_duplicates()
    .sort_values(['phase_roi', 'amp_roi'])
    .reset_index(drop=True)
)
if len(pair_keys) == 0:
    raise ValueError('No unique PAC ROI pairs found in the combined PAC table.')

pair_subject_tables = {}
pair_subject_long_rows = []
for pair in pair_keys.itertuples(index=False):
    phase_roi = str(pair.phase_roi)
    amp_roi = str(pair.amp_roi)
    pair_label = f'{phase_roi}_{amp_roi}'
    pair_tbl = pac_tbl[(pac_tbl['phase_roi'] == phase_roi) & (pac_tbl['amp_roi'] == amp_roi)].copy()
    if len(pair_tbl) == 0:
        raise ValueError(f'No cue_on {pair_label} PAC rows found after filtering.')

    pac_overall_tbl = (
        pair_tbl.groupby('participant', as_index=False)['trial_pac_z']
        .mean()
        .rename(columns={'participant': 'Subject', 'trial_pac_z': f'{pair_label}_PAC_overall_cue_on'})
    )

    pac_rpe_tbl = (
        pair_tbl.groupby(['participant', 'rpe_cat'], as_index=False)['trial_pac_z']
        .mean()
        .pivot(index='participant', columns='rpe_cat', values='trial_pac_z')
        .reset_index()
        .rename(columns={
            'participant': 'Subject',
            'neg': f'{pair_label}_PAC_neg_cue_on',
            'neutral': f'{pair_label}_PAC_neutral_cue_on',
            'pos': f'{pair_label}_PAC_pos_cue_on',
        })
    )

    required_pac_cols = [
        'Subject',
        f'{pair_label}_PAC_neg_cue_on',
        f'{pair_label}_PAC_neutral_cue_on',
        f'{pair_label}_PAC_pos_cue_on',
    ]
    assert_required_columns(pac_rpe_tbl, required_pac_cols, f'PAC RPE table ({pair_label})')

    pair_subject_tbl = deterministic_inner_join(pac_overall_tbl, pac_rpe_tbl, keys=['Subject'])
    pair_subject_tables[pair_label] = pair_subject_tbl

    pair_long = pair_subject_tbl.rename(columns={
        f'{pair_label}_PAC_overall_cue_on': 'PAC_overall_cue_on',
        f'{pair_label}_PAC_neg_cue_on': 'PAC_neg_cue_on',
        f'{pair_label}_PAC_neutral_cue_on': 'PAC_neutral_cue_on',
        f'{pair_label}_PAC_pos_cue_on': 'PAC_pos_cue_on',
    }).copy()
    pair_long['phase_roi'] = phase_roi
    pair_long['amp_roi'] = amp_roi
    pair_subject_long_rows.append(pair_long)

    print(f'Subjects in {pair_label} PAC cue_on rows:', pair_tbl['participant'].nunique())
    print(f'Subjects in {pair_label} PAC subject table:', pair_subject_tbl['Subject'].nunique())
    print(f'{pair_label} PAC RPE categories present:', sorted(pair_tbl['rpe_cat'].dropna().unique().tolist()))

pac_subject_long_tbl = pd.concat(pair_subject_long_rows, ignore_index=True)
print('PAC ROI pairs modeled:', [f'{row.phase_roi}_{row.amp_roi}' for row in pair_keys.itertuples(index=False)])
print('Rows in long PAC subject table:', len(pac_subject_long_tbl))
display(pac_subject_long_tbl.head())


In [ ]:
# ---- Merge PAC with model table pair-by-pair, run Spearman per predictor/outcome, and save outputs ----
pac_predictors = [
    'ACC_MeanAccuracy_z',
    rsa_metric_col,
    'ACC_hfa_MeanAccuracy_z',
    'HPC_theta_MeanAccuracy_z',
    'HPC_ACC_theta_gc_net_z_cue_full_pos_z',
]

pair_overlap_rows = []
pac_coef_rows = []
pac_summary_chunks = []
pac_model_tables = {}

for pair_label, pair_subject_tbl in pair_subject_tables.items():
    pac_model_pair_tbl = deterministic_inner_join(model_tbl, pair_subject_tbl, keys=['Subject'])
    if len(pac_model_pair_tbl) == 0:
        pair_overlap_rows.append({
            'pair_label': pair_label,
            'phase_roi': pair_label.split('_')[0],
            'amp_roi': pair_label.split('_')[1],
            'n_subjects': 0,
            'status': 'no_overlap',
        })
        continue

    pac_model_tables[pair_label] = pac_model_pair_tbl

    phase_roi, amp_roi = pair_label.split('_')
    pair_outcomes = [
        f'{pair_label}_PAC_neg_cue_on',
        f'{pair_label}_PAC_neutral_cue_on',
        f'{pair_label}_PAC_pos_cue_on',
        f'{pair_label}_PAC_overall_cue_on',
    ]

    pair_overlap_rows.append({
        'pair_label': pair_label,
        'phase_roi': phase_roi,
        'amp_roi': amp_roi,
        'n_subjects': int(len(pac_model_pair_tbl)),
        'status': 'overlap_ok',
    })

    missing_pac = pac_model_pair_tbl[['Subject'] + pac_predictors + pair_outcomes].isna().sum()
    print(f'Missing-value audit for {pair_label}:')
    display(missing_pac.to_frame(name='n_missing').T)

    for outcome in pair_outcomes:
        pac_split = outcome.split('_PAC_')[1]

        for predictor in pac_predictors:
            corr_tbl = pac_model_pair_tbl.dropna(subset=[predictor, outcome]).copy()
            n_subjects = len(corr_tbl)

            if n_subjects < 3:
                pac_coef_rows.append({
                    'phase_roi': phase_roi,
                    'amp_roi': amp_roi,
                    'pac_split': pac_split,
                    'pac_outcome': outcome,
                    'term': predictor,
                    'rho': np.nan,
                    'p_value': np.nan,
                    'n_subjects': int(n_subjects),
                    'status': 'too_few_points',
                })
                continue

            rho, p_value = spearmanr(
                corr_tbl[predictor].to_numpy(dtype=float),
                corr_tbl[outcome].to_numpy(dtype=float),
            )

            pac_coef_rows.append({
                'phase_roi': phase_roi,
                'amp_roi': amp_roi,
                'pac_split': pac_split,
                'pac_outcome': outcome,
                'term': predictor,
                'rho': float(rho),
                'p_value': float(p_value),
                'n_subjects': int(n_subjects),
                'status': 'ok',
            })

        pair_outcome_tbl = pd.DataFrame([
            r for r in pac_coef_rows
            if r['phase_roi'] == phase_roi
            and r['amp_roi'] == amp_roi
            and r['pac_outcome'] == outcome
        ])
        pac_summary_chunks.append(f'PAC outcome: {outcome} ({phase_roi}->{amp_roi}, {pac_split})')
        pac_summary_chunks.append('-' * 80)
        pac_summary_chunks.append(pair_outcome_tbl.to_string(index=False))
        pac_summary_chunks.append('')

pair_overlap_df = pd.DataFrame(pair_overlap_rows).sort_values(['phase_roi', 'amp_roi']).reset_index(drop=True)
print('Per-pair overlap counts:')
display(pair_overlap_df)

if ((pair_overlap_df['pair_label'] == 'HPC_ACC') & (pair_overlap_df['n_subjects'] == 15)).sum() == 0:
    raise ValueError('Expected HPC_ACC overlap to be 15 subjects, but it was not.')

pac_coef_df = pd.DataFrame(pac_coef_rows)

all_pac_model_tbl = pd.concat(
    [
        tbl.assign(phase_roi=pair.split('_')[0], amp_roi=pair.split('_')[1])
        for pair, tbl in pac_model_tables.items()
    ],
    ignore_index=True,
)

pac_outcomes = sorted([c for c in all_pac_model_tbl.columns if c.endswith('_PAC_neg_cue_on') or c.endswith('_PAC_neutral_cue_on') or c.endswith('_PAC_pos_cue_on') or c.endswith('_PAC_overall_cue_on')])
pac_corr_cols = pac_predictors + pac_outcomes
pac_corr_df = all_pac_model_tbl[pac_corr_cols].corr(method='spearman')
fig, ax = plt.subplots(1, 1, figsize=(10, 8), constrained_layout=True)
sns.heatmap(pac_corr_df, cmap='vlag', center=0.0, square=True, ax=ax)
ax.set_title('PAC Outcomes and Predictors Correlation Matrix (Spearman)')
fig.savefig(paths['pac_corr_pdf'], dpi=300)
plt.close(fig)

rho_plot_tbl = pac_coef_df[
    (pac_coef_df['pac_split'] == 'overall_cue_on')
    & np.isfinite(pac_coef_df['rho'])
].copy()
if len(rho_plot_tbl) > 0:
    rho_pivot = rho_plot_tbl.pivot_table(
        index='term',
        columns='pac_outcome',
        values='rho',
        aggfunc='mean',
    )
    fig, ax = plt.subplots(1, 1, figsize=(12, 4), constrained_layout=True)
    sns.heatmap(rho_pivot, cmap='vlag', center=0.0, ax=ax)
    ax.set_title('Spearman rho by predictor and PAC overall outcome')
    fig.savefig(paths['pac_diagnostic_pdf'], dpi=300)
    plt.close(fig)

pac_subject_long_tbl.to_csv(paths['pac_subject_csv'], index=False)
pac_coef_df.to_csv(paths['pac_coef_csv'], index=False)
with open(paths['pac_summary_txt'], 'w', encoding='utf-8') as f:
    f.write('PAC OUTCOME SPEARMAN CORRELATIONS\n')
    f.write('===============================\n\n')
    f.write('Per-pair overlap counts:\n')
    f.write(pair_overlap_df.to_string(index=False))
    f.write('\n\n')
    f.write(f'Predictors: {pac_predictors}\n')
    f.write(f'Outcomes: {pac_outcomes}\n\n')
    f.write('\n'.join(pac_summary_chunks))

print('Saved PAC subject table:', paths['pac_subject_csv'])
print('Saved PAC Spearman results:', paths['pac_coef_csv'])
print('Saved PAC Spearman summary:', paths['pac_summary_txt'])
print('Saved PAC correlation heatmap:', paths['pac_corr_pdf'])
print('Saved PAC rho heatmap:', paths['pac_diagnostic_pdf'])
display(pac_coef_df.head(20))


In [ ]:
# ---- Bar + strip: overall cue_on PAC across all phase/amp ROI pairs ----
plot_tbl = pac_subject_long_tbl.copy()
plot_tbl = plot_tbl[np.isfinite(plot_tbl['PAC_overall_cue_on'])].copy()
plot_tbl['pair_label'] = plot_tbl['phase_roi'].astype(str) + '_' + plot_tbl['amp_roi'].astype(str)

pair_order = (
    plot_tbl[['phase_roi', 'amp_roi', 'pair_label']]
    .drop_duplicates()
    .sort_values(['phase_roi', 'amp_roi'])['pair_label']
    .tolist()
)

fig_width = max(10, 1.2 * max(1, len(pair_order)))
fig, ax = plt.subplots(1, 1, figsize=(fig_width, 5), constrained_layout=True)

sns.barplot(
    data=plot_tbl,
    x='pair_label',
    y='PAC_overall_cue_on',
    order=pair_order,
    estimator=np.mean,
    errorbar=None,
    color='#9ecae1',
    ax=ax,
)

sns.stripplot(
    data=plot_tbl,
    x='pair_label',
    y='PAC_overall_cue_on',
    order=pair_order,
    color='#1f1f1f',
    alpha=0.72,
    size=5,
    jitter=0.18,
    linewidth=0.35,
    edgecolor='white',
    ax=ax,
)

ax.set_xlabel('PAC pair (phase_roi_amp_roi)')
ax.set_ylabel('PAC_overall_cue_on')
ax.set_title('Overall cue_on PAC across all ROI pairs')
ax.tick_params(axis='x', rotation=45)
ax.grid(False)
sns.despine(ax=ax, top=True, right=True)

fig.savefig(paths['pac_overall_all_pairs_bar_strip_pdf'], dpi=300)
# plt.close(fig)
print('Saved all-pairs PAC overall bar+strip plot:', paths['pac_overall_all_pairs_bar_strip_pdf'])


In [ ]:
# ---- Scatter: overall cue_on PAC vs ACC_HPC_HitRSA_cluster_mean (HPC_ACC / HPC_HPC / ACC_ACC) ----
scatter_specs = [
    ('HPC_ACC', 'pac_rsa_scatter_pdf', 'HPC-ACC overall cue_on PAC vs ACC-HPC Hit RSA cluster mean'),
    ('HPC_HPC', 'pac_rsa_scatter_hpc_hpc_pdf', 'HPC-HPC overall cue_on PAC vs ACC-HPC Hit RSA cluster mean'),
    ('ACC_ACC', 'pac_rsa_scatter_acc_acc_pdf', 'ACC-ACC overall cue_on PAC vs ACC-HPC Hit RSA cluster mean'),
]

for pair_label, output_key, title in scatter_specs:
    if pair_label not in pac_model_tables:
        raise ValueError(f'{pair_label} pair table not found in PAC model tables.')

    pair_model_tbl = pac_model_tables[pair_label]
    x_col = f'{pair_label}_PAC_overall_cue_on'
    y_col = rsa_metric_col

    plot_df = pair_model_tbl[
        np.isfinite(pair_model_tbl[x_col])
        & np.isfinite(pair_model_tbl[y_col])
    ].copy()

    if len(plot_df) < 3:
        raise ValueError(f'Need at least 3 points for {pair_label} scatter; got {len(plot_df)}.')

    x = plot_df[x_col].to_numpy(dtype=float)
    y = plot_df[y_col].to_numpy(dtype=float)

    fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)
    sns.scatterplot(x=x, y=y, ax=ax, s=55, color='#4c78a8', edgecolor='black', linewidth=0.7)

    if len(np.unique(x)) > 1:
        coeff = np.polyfit(x, y, 1)
        xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
        yfit = np.polyval(coeff, xfit)
        ax.plot(xfit, yfit, color='#222222', linewidth=1.2)

    for row in plot_df.itertuples(index=False):
        ax.text(getattr(row, x_col), getattr(row, y_col), f' {row.Subject}', fontsize=8)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(False)
    sns.despine(ax=ax, top=True, right=True)

    fig.savefig(paths[output_key], dpi=300)
    # plt.close(fig)
    print(f'Saved {pair_label} PAC vs RSA scatter plot:', paths[output_key])


In [ ]:
# ---- Spearman correlations: PAC_overall_cue_on vs ACC_HPC_HitRSA_cluster_mean ----
spearman_specs = [
    ('HPC_ACC', 'HPC_ACC_PAC_overall_cue_on'),
    ('HPC_HPC', 'HPC_HPC_PAC_overall_cue_on'),
    ('ACC_ACC', 'ACC_ACC_PAC_overall_cue_on'),
]

spearman_rows = []
y_col = rsa_metric_col
# rsa_metric_col

for pair_label, x_col in spearman_specs:
    if pair_label not in pac_model_tables:
        raise ValueError(f'{pair_label} pair table not found in PAC model tables.')

    pair_tbl = pac_model_tables[pair_label]
    corr_tbl = pair_tbl[
        np.isfinite(pair_tbl[x_col])
        & np.isfinite(pair_tbl[y_col])
    ][['Subject', x_col, y_col]].copy()

    n_subjects = len(corr_tbl)
    if n_subjects < 3:
        spearman_rows.append({
            'pair_label': pair_label,
            'x_col': x_col,
            'y_col': y_col,
            'n_subjects': n_subjects,
            'rho': np.nan,
            'p_value': np.nan,
            'status': 'too_few_points',
        })
        continue

    rho, p_value = spearmanr(corr_tbl[x_col].to_numpy(dtype=float), corr_tbl[y_col].to_numpy(dtype=float))
    spearman_rows.append({
        'pair_label': pair_label,
        'x_col': x_col,
        'y_col': y_col,
        'n_subjects': n_subjects,
        'rho': float(rho),
        'p_value': float(p_value),
        'status': 'ok',
    })

spearman_df = pd.DataFrame(spearman_rows)
print('Spearman PAC_overall_cue_on vs ACC_HPC_HitRSA_cluster_mean:')
display(spearman_df)


In [ ]:
pac_coef_df[((pac_coef_df.amp_roi=='ACC') & (pac_coef_df.phase_roi=='ACC')) | ((pac_coef_df.amp_roi=='HPC') & (pac_coef_df.phase_roi=='HPC')) | ((pac_coef_df.amp_roi=='HPC') & (pac_coef_df.phase_roi=='ACC'))]

In [ ]:
pac_coef_df[(pac_coef_df.p_value<=0.01) & (pac_coef_df.term.isin(['ACC_MeanAccuracy_z', 'ACC_HPC_HitRSA_cluster_mean_z']))]